# MultiMed — Audio → ASR → NER demo

Notebook demo cho hai cách nhập audio: upload file hoặc ghi âm microphone trong browser Colab. Không dùng để chẩn đoán; hệ thống chỉ chuyển lời nói thành transcript và trích xuất thực thể y tế.

**Lưu ý:** chọn GPU T4. Microphone yêu cầu trình duyệt cấp quyền; nếu ghi âm không hoạt động, dùng `INPUT_MODE = 'upload'`.

In [ ]:
!pip -q install -U "transformers>=4.41,<5" accelerate safetensors soundfile librosa pandas matplotlib

In [ ]:
import base64
import json
import shutil
import unicodedata
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display
from transformers import (
    AutoFeatureExtractor, AutoModelForSpeechSeq2Seq,
    AutoModelForTokenClassification, AutoProcessor,
    AutoTokenizer, pipeline,
)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
INPUT_ROOT = DRIVE_ROOT / 'MultiMed_inputs'
OUTPUT_ROOT = DRIVE_ROOT / 'MultiMed_demo_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PHOBERT_ZIP = INPUT_ROOT / 'phobert-best-seed.zip'
PHOBERT_DIR = INPUT_ROOT / 'phobert-vietmed-ner'
ASR_REPO = 'leduckhai/MultiMed-ST'
ASR_SUBFOLDER = 'asr/whisper-small-vietnamese/checkpoint-5000'
ASR_PROCESSOR_SUBFOLDER = 'asr/whisper-small-vietnamese'
XLM_REPO = 'leduckhai/VietMed-NER'
XLM_SUBFOLDER = 'xlm-roberta-base-VietMed-NER'
INPUT_MODE = 'upload'  # 'upload' or 'record'
ACTIVE_NER_MODEL = 'phobert'  # 'phobert' or 'xlm_roberta'
RUN_CORRECTION = False  # keep False for the direct production demo
TARGET_RATE = 16000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if not INPUT_ROOT.exists():
    raise FileNotFoundError('Không tìm thấy MyDrive/MultiMed_inputs')
if not PHOBERT_ZIP.exists() and not PHOBERT_DIR.exists():
    raise FileNotFoundError('Thiếu phobert-best-seed.zip hoặc thư mục PhoBERT trong MultiMed_inputs')

print('Device:', DEVICE)
print('Input mode:', INPUT_MODE)
print('NER model:', ACTIVE_NER_MODEL)

In [ ]:
# Prepare PhoBERT checkpoint from Drive.
if not PHOBERT_DIR.exists() and PHOBERT_ZIP.exists():
    extract_root = INPUT_ROOT / '_demo_phobert_extract'
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PHOBERT_ZIP) as archive:
        archive.extractall(extract_root)
    candidates = [
        p.parent for p in extract_root.rglob('config.json')
        if (p.parent / 'model.safetensors').exists() and not p.parent.name.startswith('checkpoint-')
    ]
    if not candidates:
        raise FileNotFoundError('Không tìm thấy model.safetensors trong phobert-best-seed.zip')
    PHOBERT_DIR = candidates[0]
if not PHOBERT_DIR.exists():
    raise FileNotFoundError('Thiếu phobert-best-seed.zip trong MyDrive/MultiMed_inputs')
print('PhoBERT:', PHOBERT_DIR)

In [ ]:
# Upload an audio file or record from the browser microphone.
if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Chưa chọn file audio')
    INPUT_AUDIO = Path('/content') / next(iter(uploaded))
elif INPUT_MODE == 'record':
    from google.colab import output
    record_js = r'''
    async function recordWav() {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, 10000));
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(chunks, {type: recorder.mimeType});
      const buffer = await blob.arrayBuffer();
      let binary = '';
      const bytes = new Uint8Array(buffer);
      for (let i = 0; i < bytes.length; i += 0x8000)
        binary += String.fromCharCode(...bytes.subarray(i, i + 0x8000));
      return btoa(binary);
    }
    recordWav();
    '''
    encoded_audio = output.eval_js(record_js)
    INPUT_AUDIO = Path('/content/recorded_audio.webm')
    INPUT_AUDIO.write_bytes(base64.b64decode(encoded_audio))
else:
    raise ValueError("INPUT_MODE phải là 'upload' hoặc 'record'")

print('Input audio:', INPUT_AUDIO)
display(Audio(filename=str(INPUT_AUDIO)))

In [ ]:
# Decode, convert to mono float32 and resample to 16 kHz.
try:
    waveform, original_rate = sf.read(str(INPUT_AUDIO), dtype='float32', always_2d=False)
except Exception:
    # Browser recordings are commonly WebM/Opus; librosa/audioread handles them when ffmpeg is available.
    waveform, original_rate = librosa.load(str(INPUT_AUDIO), sr=None, mono=False)
waveform = np.asarray(waveform, dtype='float32')
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1)
if len(waveform) == 0:
    raise ValueError('Audio rỗng')
if int(original_rate) != TARGET_RATE:
    waveform = librosa.resample(waveform, orig_sr=int(original_rate), target_sr=TARGET_RATE)
sample_rate = TARGET_RATE
NORMALIZED_AUDIO = Path('/content/demo_audio_16k_mono.wav')
sf.write(str(NORMALIZED_AUDIO), waveform, sample_rate)
print('Original rate:', original_rate, 'Hz')
print('Normalized:', NORMALIZED_AUDIO, 'samples:', len(waveform))
display(Audio(waveform, rate=sample_rate))

In [ ]:
# Load Whisper once; support both full AutoProcessor and direct tokenizer return values.
if not torch.cuda.is_available():
    raise RuntimeError('Demo ASR cần GPU CUDA/T4 trong Colab')
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_REPO, subfolder=ASR_SUBFOLDER, dtype=torch.float16,
    low_cpu_mem_usage=True, use_safetensors=True,
).to('cuda')
processor = AutoProcessor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
asr_tokenizer = getattr(processor, 'tokenizer', processor)
asr_feature_extractor = getattr(processor, 'feature_extractor', None)
if asr_feature_extractor is None:
    asr_feature_extractor = AutoFeatureExtractor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
asr_pipe = pipeline(
    'automatic-speech-recognition', model=asr_model, tokenizer=asr_tokenizer,
    feature_extractor=asr_feature_extractor, dtype=torch.float16, device=0,
)
asr_result = asr_pipe(
    {'raw': waveform, 'sampling_rate': sample_rate},
    generate_kwargs={'language': 'Vietnamese', 'task': 'transcribe'},
)
transcript = asr_result['text'].strip()
print('Transcript ASR:')
print(transcript)

In [ ]:
# Load selected NER model and run optional canonical correction.
NER_MAX_LENGTH = 256
ner_specs = {
    'xlm_roberta': (XLM_REPO, {'subfolder': XLM_SUBFOLDER}),
    'phobert': (str(PHOBERT_DIR), {}),
}
if ACTIVE_NER_MODEL not in ner_specs:
    raise ValueError('ACTIVE_NER_MODEL phải là phobert hoặc xlm_roberta')
ner_repo, ner_kwargs = ner_specs[ACTIVE_NER_MODEL]
ner_model = AutoModelForTokenClassification.from_pretrained(ner_repo, **ner_kwargs).to(DEVICE)
ner_tokenizer = AutoTokenizer.from_pretrained(ner_repo, **ner_kwargs)
ner_tokenizer.model_max_length = NER_MAX_LENGTH
ner_pipe = pipeline(
    'ner', model=ner_model, tokenizer=ner_tokenizer,
    aggregation_strategy='simple', device=0 if DEVICE == 'cuda' else -1,
)

# Optional rulebase input is read from Drive only when correction is enabled.
corrected_transcript = transcript
correction_changes = []
if RUN_CORRECTION:
    terms_path = INPUT_ROOT / 'giai_doan_11_text_rulebase' / 'canonical_terms.json'
    phrases_path = INPUT_ROOT / 'giai_doan_11_text_rulebase' / 'canonical_phrases.json'
    if not terms_path.exists() or not phrases_path.exists():
        raise FileNotFoundError('Thiếu canonical rulebase trong MultiMed_inputs')
    demo_terms = json.loads(terms_path.read_text(encoding='utf-8'))
    demo_phrases = json.loads(phrases_path.read_text(encoding='utf-8'))
    # Conservative demo correction: exact canonical phrase/term match after accent-insensitive key.
    def demo_key(value):
        decomposed = unicodedata.normalize('NFD', value.lower())
        return ''.join(c for c in decomposed if unicodedata.category(c) != 'Mn').replace('đ','d')
    phrase_map = {demo_key(x['text']): x['text'] for x in demo_phrases if x.get('count', 0) >= 2}
    term_map = {demo_key(x['text']): x['text'] for x in demo_terms if x.get('count', 0) >= 2}
    corrected_words = []
    for word in transcript.split():
        replacement = term_map.get(demo_key(word), word)
        corrected_words.append(replacement)
        if replacement != word: correction_changes.append({'source': word, 'target': replacement, 'rule': 'exact_canonical_term'})
    corrected_transcript = ' '.join(corrected_words)

def extract_entities(text):
    output = []
    for item in ner_pipe(text):
        label = item.get('entity_group') or item.get('entity')
        if label in {'0', 'O', 'dum'}: continue
        start, end = item.get('start'), item.get('end')
        output.append({'text': item['word'], 'label': label, 'score': round(float(item['score']), 6), 'start': int(start) if start is not None else None, 'end': int(end) if end is not None else None})
    return output
entities_direct = extract_entities(transcript)
entities_corrected = extract_entities(corrected_transcript) if RUN_CORRECTION else entities_direct
result = {
    'audio_source': str(INPUT_AUDIO), 'asr_model': f'{ASR_REPO}/{ASR_SUBFOLDER}',
    'ner_model': ACTIVE_NER_MODEL, 'correction_enabled': RUN_CORRECTION,
    'transcript': transcript, 'corrected_transcript': corrected_transcript,
    'correction_changes': correction_changes,
    'entities': entities_corrected if RUN_CORRECTION else entities_direct,
}
print('NER model:', ACTIVE_NER_MODEL)
print('Correction enabled:', RUN_CORRECTION)
print('Entities:')
display(pd.DataFrame(result['entities']))
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Save and download the demo JSON.
demo_output = OUTPUT_ROOT / 'latest_demo_result.json'
demo_output.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved to Google Drive:', demo_output)
from google.colab import files
files.download(str(demo_output))